In [ ]:
# from vllm import LLM, SamplingParams
# model_name = "Qwen/Qwen2-VL-2B-Instruct"
# llm = LLM(model=model_name)

In [ ]:
import sys
sys.path.append("data/g3/raw")
from dataset import MsgPackIterableDatasetMultiTargetWithDynLabels

In [ ]:
# image, problem, solution
import json
target_mapping = json.load(open("data/g3/raw/train/countries.json"))
ds = iter(MsgPackIterableDatasetMultiTargetWithDynLabels(
    path="data/g3/raw/train/msgpack",
    target_mapping=target_mapping,
    key_img_id="id",
    key_img_encoded="image",
    shuffle=False,
    cache_size=350000,
))

In [ ]:
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
import io
import base64
from datasets import Image

def list_to_dict(list_of_dicts):
    result = {}
    for d in list_of_dicts:
        for key, value in d.items():
            result.setdefault(key, []).append(value)
    return result

problem = "Based on visual cues, such as road signs, architecture, vegetation, and landscape, identify the most likely country where this image was taken."
country_to_id = pd.read_csv("data/g3/raw/countries.csv", skiprows=0)
country_to_id = {c["class_label"]: c["country"] for c in country_to_id.to_dict(orient="records")}
hf = []
counter = 0
for x in tqdm(ds):
    ann = {"image": x[0], "id": x[2], "solution": f"<answer>{country_to_id[x[1]]}</answer>", "problem": problem}
    hf.append(ann)
    counter += 1
    # if counter > 10:
    #     break

In [ ]:
import numpy as np
idxs = np.random.permutation(range(len(hf)))

In [ ]:
# chunk = [hf[i] for i in idxs[:10000]]
chunk = hf
chunk = list_to_dict(chunk)
chunk = Dataset.from_dict(chunk)
chunk = chunk.cast_column("image", Image())
chunk.to_parquet(f"data/g3/data/val.parquet")

In [ ]:
chunk_size = 25000
n_chunks = (len(hf) + chunk_size - 1) // chunk_size  # Calculate number of chunks

for i in tqdm(range(n_chunks)):
    start = i * chunk_size
    end = min(start + chunk_size, len(hf))
    chunk = hf[start:end]
    chunk = list_to_dict(chunk)
    chunk = Dataset.from_dict(chunk)
    chunk = chunk.cast_column("image", Image())
    chunk.to_parquet(f"data/g3/g3-streetview-300k/data/train_{str(i).zfill(3)}.parquet")

In [1]:
import datasets
x = datasets.load_dataset("data/g3/g3-streetview-300k/data")

/home/coder/miniconda3/envs/r1/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 322536 examples [00:11, 29289.59 examples/s]
Generating validation split: 3887 examples [00:00, 30996.20 examples/s]


In [2]:
len(x["train"])

322536

In [ ]:
import wandb
from tqdm import tqdm

# Create an API object
api = wandb.Api()

# Fetch the run from W&B
run = api.run(f"g-luo/huggingface/6b79ctfu")

# Get all files in this run
files = run.files()

# Filter for the files you want to delete. For example, if you only want to delete
# files whose name starts with "results_", do something like:
results_files = [f for f in files if "results" in f.name]
for f in tqdm(results_files):
    if "results_1000_3362ec8424a7c378c62d.table.json" not in f.name:
        print(f.name)
        f.delete()

In [ ]:
for i, f in enumerate(files):
  if "results_1000_3362ec8424a7c378c62d.table.json" in f.name:
    print(f)